# Week 23 — Safe exploration, LTL shielding, CBFs, robust RL

Eight graded exercises, each with a hint and a test against `project.solutions`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as ref
from project.env_grid import DangerGridWorld
env = DangerGridWorld(); env.reset(seed=0)
print(env.render_ascii())

## Exercise 1 ★ — DFA for G(¬danger)

Build the 2-state safety automaton.

> **Hint**: states `q0` (accepting) and `q_bad` (rejecting sink); `danger` sends `q0` to `q_bad` and absorbs `q_bad`.

In [ ]:
def exercise_1():
    # YOUR CODE HERE
    return ref.exercise_1()

In [ ]:
dfa = exercise_1()
assert dfa['initial'] == 'q0' and 'q_bad' not in dfa['accepting']
print(dfa)

## Exercise 2 ★ — One-step unsafe predicate

Return True iff `action a` in `state s` would enter a danger cell.

> **Hint**: dynamics are deterministic; use `env.adjacent_dangerous`.

In [ ]:
def exercise_2(env, s_idx, a):
    # YOUR CODE HERE
    return ref.exercise_2(env, s_idx, a)

In [ ]:
assert exercise_2(env, env.s2idx((1,1)), 1) == True
assert exercise_2(env, env.s2idx((0,0)), 2) == False
print('ok')

## Exercise 3 ★★ — Pre-shield filtering

Given an action preference list, return the first SAFE action.

> **Hint**: iterate through preferences; return first one whose lookahead is safe.

In [ ]:
def exercise_3(env, preferences, s_idx):
    # YOUR CODE HERE
    return ref.exercise_3(env, preferences, s_idx)

In [ ]:
out = exercise_3(env, [1, 2, 0, 3], env.s2idx((1,1)))
assert out in (0, 3)
print('chose', out)

## Exercise 4 ★★ — Shielded vs unshielded Q-learning

Train both, compare cumulative unsafe transitions.

In [ ]:
def exercise_4(seed=0, n_episodes=200):
    # YOUR CODE HERE
    return ref.exercise_4(seed=seed, n_episodes=n_episodes)

In [ ]:
res = exercise_4(seed=0, n_episodes=200)
print('shielded unsafe', res['shielded']['total_unsafe'])
print('unshielded unsafe', res['unshielded']['total_unsafe'])
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(np.cumsum(res['unshielded']['unsafe']), color=RL_PALETTE['red'], label='unshielded')
ax.plot(np.cumsum(res['shielded']['unsafe']), color=RL_PALETTE['teal'], label='shielded')
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative unsafe transitions'); ax.legend(); plt.show()

## Exercise 5 ★★ — Verify the soundness theorem

Shielded Q-learning must produce 0 unsafe transitions in a deterministic env. Verify.

In [ ]:
def exercise_5(seed=0, n_episodes=300):
    return ref.exercise_5(seed=seed, n_episodes=n_episodes)

In [ ]:
n = exercise_5(seed=0, n_episodes=300)
assert n == 0, f'shield was violated {n} times'
print('ok, zero violations under shield')

## Exercise 6 ★★ — Meaningful objective: DR vs fixed-length pendulum

Train a tiny policy gradient agent both with and without domain randomisation, evaluate at OOD lengths.

> **Hint**: this is the matching/safety hook — in a deployed system, DR is how you make a policy that survives the next distribution shift.

In [ ]:
def exercise_6(seed=0, n_epochs=15, steps_per_epoch=300):
    return ref.exercise_6(seed=seed, n_epochs=n_epochs, steps_per_epoch=steps_per_epoch)

In [ ]:
res = exercise_6(seed=0, n_epochs=15, steps_per_epoch=300)
print('DR eval:   ', res['dr']['eval'])
print('FIXED eval:', res['fixed']['eval'])
Ls = sorted(res['dr']['eval'].keys())
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(Ls, [res['dr']['eval'][L] for L in Ls], 'o-', label='DR', color=RL_PALETTE['blue'])
ax.plot(Ls, [res['fixed']['eval'][L] for L in Ls], 's--', label='fixed', color=RL_PALETTE['orange'])
ax.axvspan(0.5,1.5, color=RL_PALETTE['teal'], alpha=0.1, label='DR train support')
ax.set_xlabel('Length'); ax.set_ylabel('Eval return'); ax.legend(); plt.show()

## Exercise 7 ★★★ — CBF projection in 1D closed form

For scalar `a`, scalar `h` and `dh/da`, project `a_proposed` onto the half-space `dh/da · a >= -alpha · h`.

> **Hint**: only project when the constraint is violated; otherwise pass through.

In [ ]:
def exercise_7(a_proposed, h_val, dh_da, alpha=1.0):
    # YOUR CODE HERE
    return ref.exercise_7(a_proposed, h_val, dh_da, alpha)

In [ ]:
assert abs(exercise_7(0.5, 2.0, 1.0, 1.0) - 0.5) < 1e-9
assert abs(exercise_7(2.0, 1.0, -1.0, 1.0) - 1.0) < 1e-9
print('ok')

## Exercise 8 ★★★ — Adversarial OOD evaluation

Train a DR policy on lengths [0.5, 1.5] then evaluate at extreme lengths (0.25, 2.0). Discuss in markdown how rapidly the policy degrades.

> **Hint**: anything more than ~3× outside the DR support tends to break.

In [ ]:
def exercise_8(seed=0, n_epochs=15, steps_per_epoch=300):
    return ref.exercise_8(seed=seed, n_epochs=n_epochs, steps_per_epoch=steps_per_epoch)

In [ ]:
ev = exercise_8(seed=0)
print(ev)